# Open-Ended Evolution of Public Interest: Google Trends Analysis (2004–2025)

Companion notebook for the MSc dissertation *Open-Ended Evolution of Public Interest: A Longitudinal Computational Analysis Using Google Trends Data, 2004–2025* (Kolakaluri, Aston University, 2026), and the accompanying paper draft.

This notebook applies **Shannon Entropy** and a **novelty-detection algorithm** to 21 years of Google Trends search-interest data, testing whether public attention to technology topics behaves as an open-ended evolutionary system (Packard et al., 2019).

**Structure:**
- Part A — preliminary analysis on a 5-topic subset (AI, Blockchain, Machine Learning, Virtual Reality, Cryptocurrency)
- Part B — full analysis on the complete 10-topic set (adds Cybersecurity, NFT, Metaverse, ChatGPT, Cloud Computing)

**Data:** exported from [Google Trends](https://trends.google.com), worldwide scope, monthly resolution, January 2004–December 2025. See `data/README.md` for how to reproduce the CSV exports.


## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import entropy

print("Libraries loaded successfully!")


## Part A — Preliminary Analysis (5 Topics)

A first pass on five early-stage technology topics, used to validate the pipeline before extending to the full 10-topic set in Part B.


### A.1 Load and merge data

In [ ]:
ai = pd.read_csv("data/multiTimeline(AI).csv", skiprows=1)
bc = pd.read_csv("data/multiTimeline(BC).csv", skiprows=1)
ml = pd.read_csv("data/multiTimeline(ML).csv", skiprows=1)
vr = pd.read_csv("data/multiTimeline(VR).csv", skiprows=1)
cc = pd.read_csv("data/multiTimeline(CC).csv", skiprows=1)

print("All files loaded!")
print(ai.head())


In [ ]:
ai.columns = ["Month", "AI"]
bc.columns = ["Month", "Blockchain"]
ml.columns = ["Month", "MachineLearning"]
vr.columns = ["Month", "VirtualReality"]
cc.columns = ["Month", "Cryptocurrency"]

df = ai.copy()
df = df.merge(bc, on="Month")
df = df.merge(ml, on="Month")
df = df.merge(vr, on="Month")
df = df.merge(cc, on="Month")

df["Month"] = pd.to_datetime(df["Month"])

print("All merged successfully!")
print(df.head(10))


### A.2 Public interest over time

In [ ]:
plt.figure(figsize=(14, 6))

for col in ["AI", "Blockchain", "MachineLearning", "VirtualReality", "Cryptocurrency"]:
    plt.plot(df["Month"], df[col], label=col)

plt.title("Public Interest Over Time — Google Trends 2004 to 2025")
plt.xlabel("Year")
plt.ylabel("Search Interest (0-100)", fontsize=12, labelpad=10)
plt.yticks([0, 20, 40, 60, 80, 100], fontsize=11)
plt.legend()
plt.tight_layout()
plt.savefig("outputs/trends_chart.png", dpi=150, bbox_inches='tight')
plt.show()

print("Chart saved!")


### A.3 Shannon Entropy

For each year, each topic's share of total search interest is treated as a probability, and Shannon Entropy is computed as H = −Σ pᵢ ln(pᵢ). A small constant (0.0001) avoids log(0) for topics with zero interest in a given year.

Note: this preliminary pass uses `dropna()`, which discards any month where a topic has no recorded value. This is adequate for the 5-topic subset (all five topics are active from 2004), but is corrected to `fillna(0)` in Part B, where later-emerging topics (e.g. ChatGPT) genuinely have zero interest before they exist — see the dissertation's methodology section for the full discussion of this correction.


In [ ]:
df["Year"] = df["Month"].dt.year

# Convert all columns to numeric, forcing errors to NaN
for col in ["AI", "Blockchain", "MachineLearning", "VirtualReality", "Cryptocurrency"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop any rows with NaN values
df = df.dropna()

diversity_scores = []

for year, group in df.groupby("Year"):
    values = group[["AI", "Blockchain", "MachineLearning",
                     "VirtualReality", "Cryptocurrency"]].mean().values
    values = values + 0.0001
    score = entropy(values)
    diversity_scores.append({"Year": year, "Shannon_Entropy": score})

diversity_df = pd.DataFrame(diversity_scores)
print(diversity_df)


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(diversity_df["Year"], diversity_df["Shannon_Entropy"],
         color="blue", linewidth=2.5, marker="o", markersize=5)

plt.title("Shannon Entropy Diversity of Public Interest Over Time")
plt.xlabel("Year")
plt.ylabel("Shannon Entropy Score")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/shannon_entropy_chart.png")
plt.show()

print("Shannon Entropy chart saved!")


### A.4 Novelty detection

A topic is classified as novel in a given year if its average interest that year exceeds 10, having remained below 2 in every preceding year. The asymmetric thresholds (10 upper, 2 lower) prevent topics that merely fluctuate at moderate levels from being repeatedly flagged. In the first year of the series, any topic already active above the threshold is counted as novel by convention (it has no prior history to compare against).


In [ ]:
topics = ["AI", "Blockchain", "MachineLearning", "VirtualReality", "Cryptocurrency"]

novelty_results = []

years = sorted(df["Year"].unique())

for i, year in enumerate(years):
    novel_count = 0
    novel_topics = []

    current = df[df["Year"] == year]
    previous = df[df["Year"] < year]

    for topic in topics:
        current_avg = current[topic].mean()

        if i == 0:
            # First year of the series: an already-active topic has no
            # prior history to compare against, so it is counted as novel
            # by convention if it exceeds the threshold.
            if current_avg > 5:
                novel_count += 1
                novel_topics.append(topic)
        else:
            prev_avg = previous[topic].mean()
            # Novel if high interest now but consistently near-zero before
            if current_avg > 10 and prev_avg < 2:
                novel_count += 1
                novel_topics.append(topic)

    novelty_results.append({
        "Year": year,
        "Novel_Topics": novel_count,
        "Topics": str(novel_topics)
    })

novelty_df = pd.DataFrame(novelty_results)
print(novelty_df)


In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(novelty_df["Year"], novelty_df["Novel_Topics"],
        color="green", alpha=0.7, edgecolor="black")

plt.title("Novel Topic Emergence Per Year — Google Trends 2004 to 2025")
plt.xlabel("Year")
plt.ylabel("Number of Novel Topics")
plt.xticks(novelty_df["Year"], rotation=45)
plt.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("outputs/novelty_chart.png")
plt.show()

print("Novelty chart saved!")


## Part B — Full Analysis (10 Topics)

Extends the pipeline to the complete ten-topic set used in the dissertation's main results: Artificial Intelligence, Blockchain, Machine Learning, Virtual Reality, Cryptocurrency, Cybersecurity, NFT, Metaverse, ChatGPT and Cloud Computing.

This is the main reported analysis (Table 1–3 and Figures in the paper draft); Part A above is a preliminary validation pass on a subset.


### B.1 Load and merge all 10 topics

In [ ]:
ai  = pd.read_csv("data/multiTimeline(AI).csv",  skiprows=1)
bc  = pd.read_csv("data/multiTimeline(BC).csv",  skiprows=1)
ml  = pd.read_csv("data/multiTimeline(ML).csv",  skiprows=1)
vr  = pd.read_csv("data/multiTimeline(VR).csv",  skiprows=1)
cc  = pd.read_csv("data/multiTimeline(CC).csv",  skiprows=1)
cs  = pd.read_csv("data/multiTimeline(CS).csv",  skiprows=1)
nft = pd.read_csv("data/multiTimeline(NFT).csv", skiprows=1)
mv  = pd.read_csv("data/multiTimeline(MV).csv",  skiprows=1)
cg  = pd.read_csv("data/multiTimeline(CG).csv",  skiprows=1)
ccm = pd.read_csv("data/multiTimeline(CCM).csv", skiprows=1)

# Rename columns
ai.columns  = ["Month", "AI"]
bc.columns  = ["Month", "Blockchain"]
ml.columns  = ["Month", "MachineLearning"]
vr.columns  = ["Month", "VirtualReality"]
cc.columns  = ["Month", "Cryptocurrency"]
cs.columns  = ["Month", "Cybersecurity"]
nft.columns = ["Month", "NFT"]
mv.columns  = ["Month", "Metaverse"]
cg.columns  = ["Month", "ChatGPT"]
ccm.columns = ["Month", "CloudComputing"]

# Merge all into one dataset
df2 = ai.copy()
for frame in [bc, ml, vr, cc, cs, nft, mv, cg, ccm]:
    df2 = df2.merge(frame, on="Month")

# Convert month to datetime
df2["Month"] = pd.to_datetime(df2["Month"])

# Convert all topic columns to numeric
topics10 = ["AI", "Blockchain", "MachineLearning", "VirtualReality",
            "Cryptocurrency", "Cybersecurity", "NFT",
            "Metaverse", "ChatGPT", "CloudComputing"]

for col in topics10:
    df2[col] = pd.to_numeric(df2[col], errors="coerce")

# fillna(0), not dropna(): a topic that had not yet been coined (e.g. ChatGPT
# before Nov 2022) genuinely has zero search interest, not unknown interest.
# Using dropna() here would discard every month where any later-emerging
# topic has no value yet, artificially flattening the early entropy curve.
df2 = df2.fillna(0)
df2["Year"] = df2["Month"].dt.year

print("All 10 topics loaded!")
print(df2.head(10))


### B.2 Public interest over time, 10 topics

In [ ]:
plt.figure(figsize=(16, 7))

colors = ["blue", "orange", "green", "red", "purple",
          "brown", "pink", "gray", "olive", "cyan"]

for col, color in zip(topics10, colors):
    plt.plot(df2["Month"], df2[col], label=col, color=color, linewidth=1.5)

plt.title("Public Interest Over Time — 10 Topics — Google Trends 2004 to 2025")
plt.xlabel("Year")
plt.ylabel("Search Interest (0–100)")
plt.legend(loc="upper left", fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/trends_chart_10topics.png")
plt.show()

print("10 topic chart saved!")


### B.3 Shannon Entropy, 10 topics (main result)

In [ ]:
diversity_scores2 = []

for year, group in df2.groupby("Year"):
    values = group[topics10].mean().values
    values = values + 0.0001
    score = entropy(values)
    diversity_scores2.append({"Year": year, "Shannon_Entropy": score})

diversity_df2 = pd.DataFrame(diversity_scores2)

plt.figure(figsize=(12, 6))
plt.plot(diversity_df2["Year"], diversity_df2["Shannon_Entropy"],
         color="blue", linewidth=2.5, marker="o", markersize=5)

plt.title("Shannon Entropy Diversity — 10 Topics — Google Trends 2004 to 2025")
plt.xlabel("Year")
plt.ylabel("Shannon Entropy Score")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/shannon_entropy_10topics.png")
plt.show()

print(diversity_df2)
print("Chart saved!")


### B.4 Novelty detection, 10 topics

Applies the same novelty rule as Part A.4 (current-year average > 10, all prior years < 2) to the full ten-topic set. This produces the three novelty events reported in the paper: Blockchain & Cryptocurrency (2017), NFT & Metaverse (2021), and ChatGPT (2023).


In [ ]:
novelty_results2 = []
years2 = sorted(df2["Year"].unique())

for i, year in enumerate(years2):
    novel_count = 0
    novel_topics = []

    current = df2[df2["Year"] == year]
    previous = df2[df2["Year"] < year]

    for topic in topics10:
        current_avg = current[topic].mean()

        if i == 0:
            if current_avg > 5:
                novel_count += 1
                novel_topics.append(topic)
        else:
            prev_avg = previous[topic].mean()
            if current_avg > 10 and prev_avg < 2:
                novel_count += 1
                novel_topics.append(topic)

    novelty_results2.append({
        "Year": year,
        "Novel_Topics": novel_count,
        "Topics": str(novel_topics)
    })

novelty_df2 = pd.DataFrame(novelty_results2)
print(novelty_df2)


In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(novelty_df2["Year"], novelty_df2["Novel_Topics"],
        color="green", alpha=0.7, edgecolor="black")

plt.title("Novel Topic Emergence Per Year — 10 Topics — Google Trends 2004 to 2025")
plt.xlabel("Year")
plt.ylabel("Number of Novel Topics")
plt.xticks(novelty_df2["Year"], rotation=45)
plt.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("outputs/novelty_chart_10topics.png")
plt.show()

print("Novelty chart (10 topics) saved!")


## Summary

- **Shannon Entropy** (10-topic set) rose from 1.03 (2004) to a peak of 2.21 (2023), against a theoretical ceiling of ln(10) ≈ 2.30.
- **Three novelty events** detected: Blockchain & Cryptocurrency (2017), NFT & Metaverse (2021), ChatGPT (2023).
- **No saturation point** observed across the 2004–2025 window.

These results are consistent with the three open-ended evolution (OEE) criteria of Packard et al. (2019): ongoing novelty generation, increasing diversity over time, and no fixed upper bound on diversity. See the dissertation and paper draft (linked in the repository README) for full discussion, limitations, and the evaluation against OEE criteria.
